# Exploratory Data Analysis

Credit risk and loan default dataset. Sections: overview, missing values, target balance, numeric distributions, numeric vs target, correlation, categorical approval rates, and takeaways.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from credit_risk.data import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    load_training_data,
)

sns.set_theme(style="whitegrid", palette="muted")

applicants = load_training_data("../data/loan_risk_prediction_dataset.csv")
applicants.head()

## Overview

In [ ]:
print(f"rows: {len(applicants)}   columns: {applicants.shape[1]}")
applicants.info()

In [ ]:
applicants[NUMERIC_FEATURES].describe()

## Missing values

In [ ]:
missing = applicants.isna().mean().mul(100).sort_values(ascending=False)
missing = missing[missing > 0]

ax = sns.barplot(x=missing.values, y=missing.index, color="#c44e52")
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
ax.set(xlabel="missing (%)", ylabel="", title="Missing values by column")
plt.show()

## Target balance

In [ ]:
approval_rate = applicants[TARGET_COLUMN].mean()
print(f"approval rate: {approval_rate:.1%}   (rejections: {1 - approval_rate:.1%})")

ax = sns.countplot(data=applicants, x=TARGET_COLUMN, hue=TARGET_COLUMN, legend=False)
ax.bar_label(ax.containers[0])
ax.set(xticklabels=["rejected (0)", "approved (1)"], xlabel="", title="Class balance")
plt.show()

## Numeric distributions

In [ ]:
fig, axes = plt.subplots(1, len(NUMERIC_FEATURES), figsize=(20, 3.5))

for feature, axis in zip(NUMERIC_FEATURES, axes, strict=True):
    sns.histplot(data=applicants, x=feature, kde=True, ax=axis)
    axis.set_title(feature)
    axis.set_xlabel("")

plt.tight_layout()
plt.show()

## Numeric features by approval

In [ ]:
fig, axes = plt.subplots(1, len(NUMERIC_FEATURES), figsize=(20, 4))

for feature, axis in zip(NUMERIC_FEATURES, axes, strict=True):
    sns.boxplot(
        data=applicants, x=TARGET_COLUMN, y=feature, hue=TARGET_COLUMN, legend=False, ax=axis
    )
    axis.set_xlabel("")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(NUMERIC_FEATURES), figsize=(20, 3.5))

for feature, axis in zip(NUMERIC_FEATURES, axes, strict=True):
    sns.kdeplot(
        data=applicants, x=feature, hue=TARGET_COLUMN, common_norm=False, fill=True, ax=axis
    )
    axis.set_title(feature)
    axis.set_xlabel("")

plt.tight_layout()
plt.show()

## Correlation

In [ ]:
correlation = applicants[NUMERIC_FEATURES + [TARGET_COLUMN]].corr()
mask = np.triu(np.ones_like(correlation, dtype=bool), k=1)

sns.heatmap(correlation, mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0, square=True)
plt.title("Pearson correlation")
plt.show()

In [ ]:
correlation[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(key=abs, ascending=False)

## Approval rate by category

In [ ]:
fig, axes = plt.subplots(1, len(CATEGORICAL_FEATURES), figsize=(20, 4))

for feature, axis in zip(CATEGORICAL_FEATURES, axes, strict=True):
    rate = applicants.groupby(feature)[TARGET_COLUMN].mean().sort_values(ascending=False)
    sns.barplot(x=rate.values, y=rate.index, hue=rate.index, legend=False, ax=axis)
    axis.axvline(approval_rate, color="black", linestyle="--", linewidth=1)
    axis.set(xlabel="approval rate", ylabel="", title=feature)

plt.tight_layout()
plt.show()

## Takeaways

- The target is imbalanced, so accuracy is misleading and average precision is the headline metric.
- `Income`, `CreditScore`, and `Education` carry missing values that the pipeline imputes rather than drops.
- `CreditScore` separates the classes most strongly; the dashed line marks the overall approval rate so category effects are read against the base rate.